In [ ]:
import rdflib, cmipld
from rdflib import Graph
from rdflib import Graph, RDF, Namespace

cmipld.client.cache_clear()

In [ ]:
folder = 'https://emd.mipcvs.dev/model/'
data = {}
data = cmipld.expand(folder + 'cnrm-esm2-1e.json',depth=1)
# data['@context'] = cmipld.client.compact('https://emd.mipcvs.dev/' + '_context',depth=0)
data

In [ ]:


# --- Step 2: Load JSON-LD into RDFLib graph ---
import json

g = Graph()


for folder in 'model model_family model_component'.split():
    url = f'https://emd.mipcvs.dev/{folder}/graph.jsonld'
    print(f'Loading {url} ...')
    data = cmipld.expand(url,depth=1)
    g.parse(data=json.dumps(data),
            # base='https://emd.mipcvs.dev/',
            format="json-ld")


# Save to a local file (Turtle format is common)
g.serialize(destination="testgraph.ttl", format="turtle")
# g2.parse("graph.ttl", format="turtle") 

print(f"Loaded {len(g)} triples.\n")




In [ ]:




print(f"\n✅ Graph loaded with {len(g)} triples\n")

# --- 1️⃣ Print all triples ---
print("=== Triples ===")
for s, p, o in g:
    print(f"{s} -- {p} --> {o}")

# --- 2️⃣ List all namespaces and prefixes ---
print("\n=== Namespaces / Prefixes ===")
for prefix, namespace in g.namespaces():
    print(f"{prefix}: {namespace}")

# --- 3️⃣ Find all classes (rdf:type) ---
print("\n=== RDF Types (Classes) ===")
for s, o in g.subject_objects(RDF.type):
    print(f"{s} a {o}")

# --- 4️⃣ Count how many times each predicate appears ---
from collections import Counter
pred_counts = Counter(p for _, p, _ in g)
print("\n=== Predicate Frequency ===")
for p, count in pred_counts.items():
    print(f"{p} : {count}")

# --- 5️⃣ (Optional) List all unique subjects ---
print("\n=== Subjects ===")
for subj in set(g.subjects()):
    print(subj)

# --- 6️⃣ (Optional) List all unique predicates ---
print("\n=== Predicates ===")
for pred in set(g.predicates()):
    print(pred)

# --- 7️⃣ (Optional) List all unique objects ---
print("\n=== Objects ===")
for obj in set(g.objects()):
    print(obj)



In [ ]:
def shorten(uri):
    for i in cmipld.mapping.keys():
        if i in uri:
            uri = uri.replace(f'https://{i}.mipcvs.dev/',f'{i}:')
            return uri

In [ ]:
nodes = []
links = []
for s, p, o in g:
    print(f"{s} -- {p} --> {o}")
    if 'https' in s and 'https' in o:
        source = shorten(str(s))
        target = shorten(str(o))
        predicate = shorten(str(p))
        nodes.append({'id': source})
        nodes.append({'id': target})
        if predicate:
            links.append({'source': source, 'target': target, 'type': predicate})

In [ ]:
links

In [ ]:
# pip install igraph ipysigma

In [ ]:
import igraph as ig
from ipysigma import Sigma

In [ ]:
# # Remove duplicates from nodes
unique_nodes = {node['id']: node for node in nodes}.values()


Gr = ig.Graph(directed=True)
# Gr.add_vertices(unique_nodes)

for i, n in enumerate(unique_nodes):
    # Gr.add_vertices([n["id"] for n in nodes])
    if n['id']!= None:
        Gr.add_vertex(n['id'])
        print(n['id'])
        node = Gr.vs.find(n['id'])
        node['label'] = n['id']
        node['color'] = 'lightblue' if 'emd:' in n['id'] else 'lightorange'

for i,val in enumerate(links): 
    Gr.add_edge(val['source'], val['target'])
    Gr.es[i]['label'] = val['type']





Gr.vs[0]
Gr.vcount()


In [ ]:
from rdflib import Graph
import json

def ttl_to_igraph_dict(ttl_path):
    """
    Parse a TTL file and create an igraph-compatible dict structure
    containing only file-to-file links (where both subject and object
    are defined subjects in the graph).
    """
    g = Graph()
    g.parse(ttl_path, format='turtle')
    
    # Get all subjects that are defined in this graph (these are our "files"/nodes)
    defined_subjects = set(g.subjects())
    
    # Build node list with name as ID
    uri_to_id = {}
    nodes = []
    for uri in sorted(defined_subjects, key=str):
        uri_str = str(uri)
        # Create a short label from the URI (last part after /)
        short_name = uri_str.split('/')[-1]
        uri_to_id[uri_str] = short_name
        nodes.append({
            "id": short_name,
            "uri": uri_str
        })
    
    # Build edge list - only where both subject and object are defined subjects
    edges = []
    for subj, pred, obj in g:
        subj_str = str(subj)
        obj_str = str(obj)
        
        # Only include edges where object is also a defined subject (file-to-file link)
        if obj in defined_subjects:
            pred_str = str(pred)
            # Get short predicate name (local part)
            pred_short = pred_str.split('/')[-1].split('#')[-1]
            
            edges.append({
                "source": uri_to_id[subj_str],
                "target": uri_to_id[obj_str],
                "label": pred_short
            })
    
    # igraph-compatible format
    igraph_dict = {
        "directed": True,
        "vertices": nodes,
        "edges": edges
    }
    
    return igraph_dict


In [ ]:


result = ttl_to_igraph_dict("testgraph.ttl")


# Load into igraph

import igraph as ig
Gaa = ig.Graph.DictList(
    vertices=[{
            "label": 'a',
            "uri": 'b',
            "name":'c'
        },{
            "label": 'a2',
            "uri": 'b2',
            "name":'c2'
        }],
    edges=[{
                "source": 'c',
                "target":'c2',
                "label": 'www'
            }
    ],
    directed=True
    
    # edges link to names (id)
)

# g.write_gexf("graph.gexf")

In [ ]:
sigma = Sigma(
    Gaa,
    # node_size_metric="degree"
    # node_label="name",
    node_color="color"
)

sigma